<a href="https://colab.research.google.com/github/NABI-SNU/book/blob/main/tutorials/Session_1_Models/student/Tutorial2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; <a href="https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/NABI-SNU/book/main/tutorials/Session_1_Models/student/Tutorial2.ipynb" target="_parent"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open in Kaggle"/></a>

# Tutorial 2: Bias-Variance Trade-Off

**Session 1: Models**

**Objective:** Understand how model complexity affects train and test error.


## Tutorial Objectives

Models that are too simple can underfit. Models that are too flexible can overfit. The bias-variance trade-off helps us reason about this balance.

In this tutorial, we will compare polynomial regression models using training and test error.

By the end, you will be able to:

- Explain why training error alone is not enough for model selection.
- Compute train and test MSE for models of different complexity.
- Identify underfitting and overfitting from error patterns.
- Relate bias and variance to model choice.


In [ ]:
# Imports and shared settings
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
def plot_MSE_poly_fits(mse_train, mse_test, max_order):
  """
    Plot the MSE values for various orders of polynomial fits on the same bar
    graph

    Args:
      mse_train (ndarray): an array of MSE values for each order of polynomial fit
      over the training data
      mse_test (ndarray): an array of MSE values for each order of polynomial fit
      over the test data
      max_order (scalar): max order of polynomial fit
  """
  fig, ax = plt.subplots()
  width = .35

  ax.bar(np.arange(max_order + 1) - width / 2,
         mse_train, width, label="train MSE")
  ax.bar(np.arange(max_order + 1) + width / 2,
         mse_test , width, label="test MSE")

  ax.legend()
  ax.set(xlabel='Polynomial order', ylabel='MSE',
         title='Comparing polynomial fits')
  plt.show()


In [ ]:
def ordinary_least_squares(x, y):
  """Ordinary least squares estimator for linear regression.

  Args:
    x (ndarray): design matrix of shape (n_samples, n_regressors)
    y (ndarray): vector of measurements of shape (n_samples)

  Returns:
    ndarray: estimated parameter values of shape (n_regressors)
  """

  return np.linalg.pinv(x) @ y


def make_design_matrix(x, order):
  """Create the design matrix of inputs for use in polynomial regression

  Args:
    x (ndarray): input vector of shape (n_samples)
    order (scalar): polynomial regression order

  Returns:
    ndarray: design matrix for polynomial regression of shape (samples, order+1)
  """

  # Broadcast to shape (n x 1) so dimensions work
  if x.ndim == 1:
    x = x[:, None]

  #if x has more than one feature, we don't want multiple columns of ones so we assign
  # x^0 here
  design_matrix = np.ones((x.shape[0],1))

  # Loop through rest of degrees and stack columns
  for degree in range(1, order+1):
      design_matrix = np.hstack((design_matrix, x**degree))

  return design_matrix


def solve_poly_reg(x, y, max_order):
  """Fit a polynomial regression model for each order 0 through max_order.

  Args:
    x (ndarray): input vector of shape (n_samples)
    y (ndarray): vector of measurements of shape (n_samples)
    max_order (scalar): max order for polynomial fits

  Returns:
    dict: fitted weights for each polynomial model (dict key is order)
  """

  # Create a dictionary with polynomial order as keys, and np array of theta
  # (weights) as the values
  theta_hats = {}

  # Loop over polynomial orders from 0 through max_order
  for order in range(max_order+1):

    X = make_design_matrix(x, order)
    this_theta = ordinary_least_squares(X, y)

    theta_hats[order] = this_theta

  return theta_hats


## Train vs Test Data


The data used for the fitting procedure for a given model is the **training data**. In Tutorial 1, we computed MSE on the training data of our polynomial regression models and compared training MSE across models. An additional important type of data is **test data**. This is held-out data that is not used in any way during the fitting procedure. When fitting models, we often want to consider both the train error, the quality of prediction on the training data, and the test error, the quality of prediction on held-out data.


We will generate noisy data from a quadratic relationship and then fit polynomial regression models to it. We will also generate separate test data so we can see how each model generalizes beyond the examples used for fitting. To accomplish this, we will generate test inputs $x$ from a wider range of values ([-3, 3]). We then plot the train and test data together.

In [ ]:
### Generate training data
np.random.seed(0)
n_train_samples = 50
x_train = np.random.uniform(-2, 2.5, n_train_samples)  # sample uniformly over [-2, 2.5)
noise = np.random.randn(n_train_samples)  # sample from a standard normal distribution
y_train = x_train**2 - x_train - 2 + noise

### Generate testing data
n_test_samples = 20
x_test = np.random.uniform(-3, 3, n_test_samples)  # sample uniformly over [-3, 3)
noise = np.random.randn(n_test_samples)  # sample from a standard normal distribution
y_test = x_test**2 - x_test - 2 + noise

## Plot both train and test data
fig, ax = plt.subplots()
plt.title('Training & Test Data')
plt.plot(x_train, y_train, '.', markersize=15, label='Training')
plt.plot(x_test, y_test, 'g+', markersize=15, label='Test')
plt.legend()
plt.xlabel('x')
plt.ylabel('y');


## Bias-Variance Trade-Off

Finding a good model can be difficult. One of the most important concepts to keep in mind when modeling is the **bias-variance trade-off**.

**Bias** is the systematic difference between the model's average prediction and the true relationship we are trying to predict. High-bias models are usually too simple, so they underfit.

**Variance** is the sensitivity of the model's prediction to the particular training dataset. High-variance models can fit the training data very closely, but they often generalize poorly to held-out data.

In essence:

- High bias, low variance models have high train and test error.
- Low bias, high variance models have low train error and high test error.
- Low bias, low variance models have low train and test error.

In this section, we will see the bias-variance trade-off in action with polynomial regression models of different orders.

![bias-variance](https://www.cs.cornell.edu/courses/cs4780/2018fa/lectures/images/bias_variance/bullseye.png)


We will first fit polynomial regression models of orders 0-5 on our simulated training data. The helper functions above define the design matrix, OLS estimator, and polynomial fitting routine needed for this step.

In [ ]:
max_order = 5
theta_hats = solve_poly_reg(x_train, y_train, max_order)

## Exercise: Compute and Compare Train vs Test Error

We will use MSE as our error metric again. Compute MSE on training data ($x_{train}, y_{train}$) and test data ($x_{test}, y_{test}$) for each polynomial regression model (orders 0-5). The setup section defines `make_design_matrix` and `evaluate_poly_reg` for your use.

*Please think about it after completing the exercise before reading the following text. Do you think the order 0 model has high or low bias? High or low variance? How about the order 5 model?*

In [ ]:
def evaluate_poly_reg(x, y, theta_hats, max_order):
  """Evaluates MSE of polynomial regression models on data.

  Args:
    x (ndarray): input vector of shape (n_samples)
    y (ndarray): vector of measurements of shape (n_samples)
    theta_hats (dict): fitted weights for each polynomial model (dict key is order)
    max_order (scalar): max order of polynomial fit

  Returns:
    ndarray: mean squared error for each order, shape (max_order + 1)
  """

  mse = np.zeros((max_order + 1))
  for order in range(0, max_order + 1):
    X_design = make_design_matrix(x, order)
    y_hat = np.dot(X_design, theta_hats[order])
    residuals = y - y_hat
    mse[order] = np.mean(residuals ** 2)

  return mse


In [ ]:
def compute_mse(x_train, x_test, y_train, y_test, theta_hats, max_order):
  """Compute MSE on training data and test data.

  Args:
    x_train (ndarray): training data input vector of shape (n_samples)
    x_test (ndarray): test data input vector of shape (n_samples)
    y_train (ndarray): training vector of measurements of shape (n_samples)
    y_test (ndarray): test vector of measurements of shape (n_samples)
    theta_hats (dict): fitted weights for each polynomial model (dict key is order)
    max_order (scalar): max order of polynomial fit

  Returns:
    ndarray, ndarray: MSE error on training data and test data for each order
  """

  #######################################################
  ## TODO for students: calculate MSE error for both sets
  ## Hint: use evaluate_poly_reg for train and test data
  # Fill out function and remove
  raise NotImplementedError("Student exercise: calculate MSE for train and test set")
  #######################################################

  mse_train = ...
  mse_test = ...

  return mse_train, mse_test


# Compute train and test MSE
mse_train, mse_test = compute_mse(x_train, x_test, y_train, y_test, theta_hats, max_order)

# Visualize
plot_MSE_poly_fits(mse_train, mse_test, max_order)


As we can see from the plot above, more complex models (higher order polynomials) have lower MSE for training data. The overly simplified models (orders 0 and 1) have high MSE on the training data. As we add complexity to the model, we go from high bias to low bias.

The MSE on test data follows a different pattern. The best test MSE is for an order 2 model - this makes sense as the data was generated with an order 2 model. Both simpler models and more complex models have higher test MSE.

So to recap:

Order 0 model: High bias, low variance

Order 5 model: Low bias, high variance

Order 2 model: Just right, low bias, low variance


## Bonus Exercise: Proof of Bias-Variance Decomposition

Prove the bias-variance decomposition for MSE:

\begin{equation}
\mathbb{E}_{x}\left[\left(y-\hat{y}(x ; \theta)\right)^{2}\right]=\left(\operatorname{Bias}_{x}[\hat{y}(x ; \theta)]\right)^{2}+\operatorname{Var}_{x}[\hat{y}(x ; \theta)]+\sigma^{2}.
\end{equation}

where

\begin{equation}
\operatorname{Bias}_{x}[\hat{y}(x ; \theta)]=\mathbb{E}_{x}[\hat{y}(x ; \theta)]-y
\end{equation}

and

\begin{equation}
\operatorname{Var}_{x}[\hat{y}(x ; \theta)]=\mathbb{E}_{x}\left[\hat{y}(x ; \theta)^{2}\right]-\mathbb{E}_{x}[\hat{y}(x ; \theta)]^{2}.
\end{equation}

Hint: use the identity

\begin{equation}
\operatorname{Var}[X]=\mathbb{E}\left[X^{2}\right]-\mathbb{E}[X]^{2}.
\end{equation}